<a href="https://www.kaggle.com/code/nagapranayimmadi/proteus-arc?scriptVersionId=327172841" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
# ------ All of the Normal Imports Required for this Algorithm -------
# __________________________________________________________________

# Os-Related
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Tkinter-Related
import tkinter as tk
from tkinter import filedialog

# mL-related
import tensorflow as tf
import numpy as np 
import pandas as pd

# Math Related
import math
import random
from itertools import combinations
import statistics
import networkx as nx 
import matplotlib.pyplot as plt 

#Scipy related imports
import scipy.io
from scipy.io import loadmat
from scipy.io.matlab import MatReadError
from scipy.signal import hilbert, butter, filtfilt, welch, iirnotch 

# EEG Processing-Related
import mne

import copy

# Torch-Related Imports
import torch # 1. Install pre-compiled CPU binaries for your exact PyTorch version
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import normalized_mutual_info_score
import warnings

# Distance-Correlation Related
!pip install dcor
import dcor
from dcor import DistanceCovarianceMethod

# Memory and Notebook Related
from IPython.display import clear_output
import gc 

# __________________________________________________________________
# Clears the Output of the Imports to make the notebook more clean 
clear_output() 

In [1]:
# ------------- Dependencies that Need to Be Instaled ------------------
# __________________________________________________________________

!pip install -q torch 
!pip install torch-scatter torch-sparse -f https://pyg.org
!pip install torch-geometric-temporal
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git


Looking in links: https://pyg.org
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 2.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677287 sha256=24f768aa1f7720869ba718b545b3f2c16eaf9aff1b3f9f1dc2566103f9724102
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
  Created wheel for torch-sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=1262259 sha256=bbd1637b1ad2022b6c281ad662aca8134639b846fa25cfd09ccecd4eb1591b49
  Stored in directory: /root/.cache/pip/wheels/71/fa/21/bd1d78ce1629aec4ecc924a63b82f6949dda484b6321eac6f2
Successfully built torch-scatter torch-sparse
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [3]:
# ------------- Machine Learning Related Imports ------------------
# __________________________________________________________________

# Torch_Geometric Related
import torch_geometric

from torch_geometric.data import Data

from torch_geometric.loader import DataLoader

from torch_geometric.nn import (
    GCNConv,
    GraphConv,
    global_mean_pool,
    global_max_pool,
    global_add_pool
)

from torch.nn import Linear, ReLU

from torch_geometric.nn import Sequential, GCNConv

# Testing Statements that should be utilized if errors are appearing in this segment of the imports
# print(torch.__version__)
# print(torch_geometric.__version__)

from torch_geometric_temporal.signal import (
    StaticGraphTemporalSignal,
    DynamicGraphTemporalSignal,
    DynamicGraphStaticSignal
)
from torch_geometric_temporal.signal import temporal_signal_split


from torch_geometric_temporal.nn.recurrent import (
    GConvGRU,
    DCRNN
)

from torch_geometric.nn import MessagePassing 

# __________________________________________________________________
# Clears the Output of the Imports to make the notebook more clean 
clear_output() 

ModuleNotFoundError: No module named 'torch_geometric'

# SEGMENT 1: FULL EDGE-BY-EDGE ALGORITHM

# RANDOMIZATION SEGMENT TO BE IMPLEMENTED LATER

# Step 1a. Data Processing / Deciding which Data to use

In terms of actually processing the EEG data, removing all of the noise, etc

In [4]:
data_input = input('what data are you using? (enter "N" for new data, use "P" for premade data to test ').upper()
# factoring time out of the algorithm (TEMPORARILY MAY CHANGE)

if data_input == "N": 
    # note this will only work in the API version which will be used in the interface
    # also, update this segment when you are making the inferace
    print("Now you will have to enter your data")
    def openFile():
        filepath = filedialog.askopenfilepath()
        file = open(filepath, 'r')
        reading_file = pd.read_csv('filepath')
        
    window = tk()
    button = Button(text="Open", command=openFile)
    button.pack()
    window.mainloop()
    """
    Also note, this notebook is rather specialized for pre-inputted data, however it can work on new
    data inputs as well
    """

# to use the data that is already in the algorithm for testing purposes
if data_input == "P":
    print("Using the dataset for this algorithm that is in the code to test")
    # In a code cell below, if the data input was chosen as P, the dataset will load. 

# to make the user know they inputted an invalid input
elif data_input != "P" and data_input !="N": 
    print("You did not enter a valid input, retry")

what data are you using? (enter "N" for new data, use "P" for premade data to test  p


Using the dataset for this algorithm that is in the code to test


# Step 1b. Class Turning the Data into the form of a Dictionary which can easily be used

In [5]:
# turning the data into dictionaries function, turning the data into dictionaires allows for easier and further processing. 
class node_to_dict: 
    def __init__(self, node_input):
        self.node_input = node_input 
    def dict(self):
        node_dict = {
            col: self.node_input[col].to_numpy()
            for col in self.node_input.columns }
        return node_dict

# Step 1c. Memory Optimization Function(Makes the values of the dataframe more Memory Efficient)

In [6]:
"""
This segment of code helps reduce the overall memory usage in the dataframe, and allows the later code
to run significantly faster, without sacrificing accuracy
Code Taken from Source: https://www.kaggle.com/discussions/questions-and-answers/405504, individual who
uploaded = Stepan Tikhonov.
When memory optimizations are specialized and created in-depth for this algorithm, a different mode of
implementation will be utilized. 
""" 

def reduce_memory_usage(df):
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if str(col_type)[:5] == "float":
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.finfo("f2").min and c_max < np.finfo("f2").max:
                df[col] = df[col].astype(np.float16)
            elif c_min > np.finfo("f4").min and c_max < np.finfo("f4").max:
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)
        elif str(col_type)[:3] == "int":
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min > np.iinfo("i1").min and c_max < np.iinfo("i1").max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo("i2").min and c_max < np.iinfo("i2").max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo("i4").min and c_max < np.iinfo("i4").max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif col == "timestamp":
            df[col] = pd.to_datetime(df[col])
        elif str(col_type)[:8] != "datetime":
            df[col] = df[col].astype("category")
    end_mem = df.memory_usage().sum() / 1024 ** 2
    # print("Memory usage reduced on:", round(start_mem - end_mem, 2), "Mb ", "or ", round(100 * (start_mem - end_mem) / start_mem), "%")
    return df

# Step 1d. Using the Premade Openneuro Dataset for Processing

In [7]:
"""
This segment means if the person chose premade data, the OpenNeuro Alzheimer's Dataset will be loaded.
Right now, there is not as much support for custom dataset input, however it will become supported considering
a good portion of the infrastructure is created in this code.
"""

if data_input == "P":
    # makig a directory for the csv's
    output_dir = "/kaggle/working/eeg_csv_outputs"
    os.makedirs(output_dir, exist_ok=True)
    # Using the Pre-Launched OpenNeuro Dataset
    openneuro_dataset = []
    for path in range(1, 89):
        file_path = f"/kaggle/input/datasets/nagapranayimmadi/openneuro-filtered-data/sub-{path:03d}_task-eyesclosed_eeg.set"
        openneuro_dataset.append(file_path)

    # Load .set file into csv usable format
    
    x = loadmat("/kaggle/input/datasets/nagapranayimmadi/openneuro-filtered-data/sub-001_task-eyesclosed_eeg.set")
    csv_openneuro_dataset = {}
    counter = 1
    for csv_convert in openneuro_dataset:
        try:     
            raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)
            # address the boundaries - DO NOW
    
            # Turning the set EEG reference into an average instead of A1/A2 Mastoid Electrodes
            raw = raw.set_eeg_reference(ref_channels='average')
            
            # Extract channel/node names
            channels = raw.ch_names
            
            # Convert to CSV format and save them in the file dir - TODO
            df = pd.DataFrame(raw.get_data().T.astype(np.float32), columns=(channels)) 
            del raw # now raw is taken away from the memory
            df = reduce_memory_usage(df)

            # NEED TO ADD BACK TO OPENNEURO DATASET CODE SEGMENT ONCE CSV'S ARE PRODUCED
            csv_openneuro_dataset.update({ f"openneuro_patient{counter}" : df})
            del df 
            counter = counter + 1 
            # print(csv_openneuro_dataset[0].head())
            
            # this creates the csv's
            # unique_filename = f"{output_dir}/openneuro_patient_{counter}.csv"
            # Save CSV
            # df.to_csv(unique_filename, index=False)
        except MatReadError as e:
            pass

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


/tmp/ipykernel_57/1718439081.py:24: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


/tmp/ipykernel_57/1718439081.py:24: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


...

/tmp/ipykernel_57/1718439081.py:24: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


/tmp/ipykernel_57/1718439081.py:24: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)


EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


/tmp/ipykernel_57/1718439081.py:24: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(csv_convert, verbose=False, preload=True)


In [8]:
"""
Continuation of the last code segment:
A loop to turn each dataset element into a dictionary. Now, the data is processed for the next step. 
The elements of this dict are looped through, with different functions of the graph_construction class. This allows the 
individual patient based data from this dataset to be transformed into ideal forms which can be interpreted as temporal graphs
over specific snapshots. These snapshots, over a full temporal graph act as an input for the GCN.  
"""

list_of_openneuro_nodes = []
for to_dict in csv_openneuro_dataset:
    dict_key = to_dict
    dict_key : node_to_dict = node_to_dict(csv_openneuro_dataset.get(to_dict))
    result = dict_key.dict()
    del dict_key
    gc.collect() 
    list_of_openneuro_nodes.append(result) #causing the memory leak 
    del result
    gc.collect()
del csv_openneuro_dataset
gc.collect()

0

# Step 1e. Function to use PLI for Node Connectivity(Edges)

In [9]:
"""
PLI Quantifies the functional connectivity between 2 different nodes, which analyzes the non-zero phase differences between 2 nodes.
This is the edge function that will be used later on to compute edges
"""

def calculate_pli(sig1, sig2, fs=500, fmin=0.5, fmax=45.0):
    """
    this function calculates the Phase Lag Index(PLI) between two signals.
    PLI ignores zero-lag (volume conduction) correlations aswell.
    """
    
    # 1. Extracting the analytical signal using the Hilbert Transformation
    analytic_sig1 = hilbert(sig1)
    analytic_sig2 = hilbert(sig2)
    
    # 2. Extracting the instantaneous phase of both signals
    phase1 = np.angle(analytic_sig1)
    phase2 = np.angle(analytic_sig2)
    
    # 3. Calculating the phase difference between the two signals
    phase_diff = phase1 - phase2
    
    # 4. Calculate PLI:
    # The sine function forces phase differences of 0 and 180 degrees (volume conduction) to 0.
    pli_value = np.abs(np.mean(np.sign(np.sin(phase_diff))))
    return pli_value

# Step 1f. Preprocessing of Data (CURRENTLY NOT DONE, WILL BE DONE LATER)

In [14]:
# Dataset Memory, RAM-Optimization, and CPU-Optimization will be implemented once the algorithm is complete.

# Full EEG Processing of the Data based on Known information, the EEG Machine, Etc.
# This will make the EEG data fully suitable for use. 
"""
Processing Steps Required based on filtering which was NOT done for the data. 
Reference for Advanced Data Processing -> https://github.com/DL4mHealth/LEAD, https://mne.tools/stable/index.html 
1. Removal of non-EEG channels
2. Notch and Band-Pass Filtering
3. Average re-referencing
4. Artifact Removal
5. Channel Alignment 
6. Frequency Alignment
7. Z-score normalization 
"""
# 1. Removal of non-EEG channels

# 2. Butterworth Filtering

butterworth_ask = input("Is your data butterworth-filtered? (Y/N) ").upper()
if input == N:
    # butterworth filtering
    pass
else: 
    print("Moving onto the next filtering round!")

# 3. Notch and Band-Pass Filtering
    
notch_ask = input("Is your data notch-filtered? (Y/N) ").lower()
if input == N:
    # filtering
    pass
else: 
    print("Moving onto the next filtering round!")

bandpass_ask = input("Is your data band-pass filtered? (Y/N) ").upper()
if input == N:
    # filtering
    pass
else: 
    print("Moving onto the next filtering round!")
    
# 4. Average re-referencing
rerefavg_ask = input("Has there been avg re-referencing done on your data? (Y/N) ").upper()
if input == N:
    # filtering
    pass
else: 
    print("Moving onto the next filtering round!")
    
# 5. Artifact Removal
artifact_ask = input("Has there been artifact removal from your data? (Y/N) ").upper()
if input == N:
    # filtering
    pass
else: 
    print("Moving onto the next filtering round!")
    
# 6. Channel Alignment 
chosen_columns = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]

# if data.column is not in chosen_columns:
    # remove that column

# 7. Frequency Alignment
freqalign_ask = input("Has there been frequency alignment done on your data? (Y/N) ").upper()
if input == N:
    # filtering
    pass
else: 
    print("Moving onto the next filtering round!")
    
# 8. Z-score normalization 
zscorenorm_ask = input("Has there been z-score normalization done on your data? (Y/N) ").upper() 
if input == N:
    # filtering
    pass
else: 
    print("Moving onto the next filtering round!")

print("The data is now fully filtered!")

KeyboardInterrupt: Interrupted by user

# Step 2. Class Graph_Construction - This class helps produce Edge Values, DCOR Values, Epochs, and other Core Data. 

<p align="center">
  <img src="https://i.sstatic.net/ZXXjJ.png" width="50%">
</p>

In [10]:
class graph_construction: 
    # These are the class-based variables that are appended with the key important features required for these functions. 
    connectivity_dict = {} # This is for creating the connectivity dict in the form { edge(such as Fp1, Fp2) : Edge Value}
    edgetoedge_dict = {} # This is for creating the edge dict in the form { dcor for example ((Fp1, Fp2), (F3, Pz)) : DCOR value}
    timestamp_wise_node_features = {} # This is for creating the timestamp wise node features that can be filtered

    def __init__(self, node_list): 
        self.node_list = node_list 
        self.list_of_nodes = list(node_list)
        self.df = pd.DataFrame(columns=self.list_of_nodes, index=self.list_of_nodes)
        self.node_list = copy.deepcopy(node_list)
        self.chunks_length = None
    
    def create_edges(self):
        # note, edge weights are temporal
        # This function is for creating all of the edges
        """
        Divide the dataset into individual components like [ list_of_openneuro_nodes[0] ] , and then from there, we transform
        the data by keeping the keys the same, but dividing the values an x number of times, while keeping the keys the same
        . This is to get temporal lists of the edges created over time from the nodes values, and also get the Distance
        Correlation values in def distance_correlation, or any other edge connectivity values which require a temporal aspect. 
        """
        for temporal_nodes in self.node_list: 
            node_pre = self.node_list.get(temporal_nodes)
            chunks = [node_pre[x:x+2000] for x in range(0, len(node_pre), 2000)]
            if len(chunks[-1]) != 2000:
                del chunks[-1]
            self.node_list[temporal_nodes] = chunks
        self.chunks_length = len(chunks)
        
        for node1, node2 in combinations(self.node_list, 2): # the larger loop for the entirety of all edge connections 
            z_list = [] 
            for length in range (0, self.chunks_length, 1):
                # Pearson Connectivity Formula
                # key for the key-value pair of connectivity values
                key_of_connectivity = (str(node1), str(node2))
                # individual edge creation inside this segment 
                x = (self.node_list[node1][length])
                y = (self.node_list[node2][length])
                z = calculate_pli(x, y, fs=500)
                z_list.append(z) 
            # creating the connectivity dictionary
            graph_construction.connectivity_dict.update({key_of_connectivity : z_list}) 

    def node_features_creation(self): # this part is for getting node features 
        # note: node features are temporal 
        # This function is for creating all of the temporal node features
        """
        The node features created in this segment for each timeframe are:
        1. Mean Voltage 
        2. Band Frequency 
            - Alpha Power
            - Beta Power
            - Theta Power
            - Delta Power
            - Gamma Power
        3. Variance
        Additionally, here in node_values_creation, we will also get the chunkwise node features. 
        """
        # looping chunkwise to get temporal node features
        for chunk_node in range(0, self.chunks_length, 1):
            parent_node_features_dict = {} 
            
            for values in self.node_list.keys(): 

                # 1. for node mean values over time
                node_mean = statistics.mean(self.node_list[values][chunk_node]) 
                
                # 2. PSD and Band Frequencies
                # saving all of the band frequencies 
                chunked_node_features = []
                alpha = []
                beta = []
                theta = []
                delta = []
                gamma = []
                
                # for PSD(power spectral densities) over time
                freqs, psd = welch(np.array(self.node_list[values][chunk_node]), fs=500, nperseg=500, noverlap=250, nfft=1000)
                
                # Calculating the band frequencies from the PSD values   
                delta_mask = (freqs >= 0.5) & (freqs < 4)
                theta_mask = (freqs >= 4) & (freqs < 8)
                alpha_mask = (freqs >= 8) & (freqs < 13)
                beta_mask  = (freqs >= 13) & (freqs < 30)
                gamma_mask = (freqs >= 30) & (freqs < 45)

                # Calculating the means of the Frequencies bands(if present) to act as node features
                # the np.trapezoid is meant to find the area itself under the graph, which is much more reliable for the band frequency
                #
                total_mask = (freqs >= 0.5) & (freqs < 45)
                total_band_power = np.trapezoid(psd[total_mask], freqs[total_mask])
                if any(alpha_mask):
                    alpha_band_power = np.trapezoid(psd[alpha_mask], freqs[alpha_mask])
                    alpha_rbp = alpha_band_power / total_band_power
                if any(beta_mask):
                    beta_band_power = np.trapezoid(psd[beta_mask], freqs[beta_mask]) # first variable for RBP(Relative Band Power)
                    beta_rbp = beta_band_power / total_band_power
                if any(theta_mask):
                    theta_band_power = np.trapezoid(psd[theta_mask], freqs[theta_mask])
                    theta_rbp = theta_band_power / total_band_power
                if any(delta_mask):
                    delta_band_power = np.trapezoid(psd[delta_mask], freqs[delta_mask])
                    delta_rbp = delta_band_power / total_band_power
                if any(gamma_mask):
                    gamma_band_power = np.trapezoid(psd[gamma_mask], freqs[gamma_mask])
                    gamma_rbp = gamma_band_power / total_band_power

                # ------ print statements for testing ------------- 
                # print(f"-------------- CHUNK {chunk_node} : NODE {values} ------------------")
                # print("Relative Alpha Band Power: ", alpha_rbp)
                # print("Relative Beta Band Power: ", beta_rbp)
                # print("Relative Theta Band Power: ", theta_rbp)
                # print("Relative Delta Band Power: ", delta_rbp)
                # print("Relative Gamma Band Power: ", gamma_rbp)

                # 3. for node variance over time
                node_variance = statistics.variance(self.node_list[values][chunk_node])

                # all of the node features for one node, chunkwise 
                all_node_features = (node_mean, alpha_rbp, beta_rbp, theta_rbp, delta_rbp, gamma_rbp, node_variance)
                all_node_features = torch.tensor(all_node_features)
                # Form of Node Tensor-Wise 
                chunked_node_features.extend(list(all_node_features))
                
                parent_node_features_dict.update( { f"{values} for chunk {chunk_node}" : chunked_node_features} )
            graph_construction.timestamp_wise_node_features.update( {f"All node values for chunk {chunk_node}" : parent_node_features_dict} )   
            # print(graph_construction.timestamp_wise_node_features["All node values for chunk 0"])
            # print(self.node_list["Fp1"][0:2])

    def distance_correlation(self): # this part creates the distance correlation
        # note: distance correlation is static
        for edge1, edge2 in combinations(graph_construction.connectivity_dict, 2):
            edge_keys = []
            value_a = np.array(graph_construction.connectivity_dict[edge1], dtype=np.float64)
            value_b = np.array(graph_construction.connectivity_dict[edge2], dtype=np.float64)
            result = dcor.distance_correlation(value_a, value_b, method=DistanceCovarianceMethod.AUTO)
            edge_key = (edge1, edge2)    
            graph_construction.edgetoedge_dict.update({ edge_key : result})
        # print("-----TOTAL DCOR VALUES OVER CHUNKS-----")
        # print(graph_construction.edgetoedge_dict)
        # print(graph_construction.connectivity_dict) -> WILL ONLY BE UNCOMMENTED FOR TESTING PURPOSES  

# Step 2a. Defining Y values and corrospondence

In [14]:
HC = 0   # Control (C)
FTD = 1  # Frontotemporal Dementia (F)
AD = 2   # Alzheimer's Disease (A)

y_dict = {
    # Alzheimer's Disease
    "sub-001": 2, "sub-002": 2, "sub-003": 2, "sub-004": 2,
    "sub-005": 2, "sub-006": 2, "sub-007": 2, "sub-008": 2,
    "sub-009": 2, "sub-010": 2, "sub-011": 2, "sub-012": 2,
    "sub-013": 2, "sub-014": 2, "sub-015": 2, "sub-016": 2,
    "sub-017": 2, "sub-018": 2, "sub-019": 2, "sub-020": 2,
    "sub-021": 2, "sub-022": 2, "sub-023": 2, "sub-024": 2,
    "sub-025": 2, "sub-026": 2, "sub-027": 2, "sub-028": 2,
    "sub-029": 2, "sub-030": 2, "sub-031": 2, "sub-032": 2,
    "sub-033": 2, "sub-034": 2, "sub-035": 2, "sub-036": 2,

    # Healthy Controls
    "sub-037": 0, "sub-038": 0, "sub-039": 0, "sub-040": 0,
    "sub-041": 0, "sub-042": 0, "sub-043": 0, "sub-044": 0,
    "sub-045": 0, "sub-046": 0, "sub-047": 0, "sub-048": 0,
    "sub-049": 0, "sub-050": 0, "sub-051": 0, "sub-052": 0,
    "sub-053": 0, "sub-054": 0, "sub-055": 0, "sub-056": 0,
    "sub-057": 0, "sub-058": 0, "sub-059": 0, "sub-060": 0,
    "sub-061": 0, "sub-062": 0, "sub-063": 0, "sub-064": 0,
    "sub-065": 0,

    # Frontotemporal Dementia
    "sub-066": 1, "sub-067": 1, "sub-068": 1, "sub-069": 1,
    "sub-070": 1, "sub-071": 1, "sub-072": 1, "sub-073": 1,
    "sub-074": 1, "sub-075": 1, "sub-076": 1, "sub-077": 1,
    "sub-078": 1, "sub-079": 1, "sub-080": 1, "sub-081": 1,
    "sub-082": 1, "sub-083": 1, "sub-084": 1, "sub-085": 1,
    "sub-086": 1, "sub-087": 1, "sub-088": 1,
}

# Step 2b. Creating a custom PyG Temporal Data Iterator Object

In [15]:
# custom PyG Temporal class that adds more parameters to StaticGraphTemporalSignal 
class StaticGraphTemporalObject(StaticGraphTemporalSignal):
    def __init__(self, edge_index, edge_weight, features, y, dcor_index, dcor_weight):
        super().__init__(edge_index, edge_weight, features,  y)
        #self.features = features
        #self.edge_index = edge_index
        #self.edge_weight = edge_weight
        #self.y = y
        self.dcor_index = dcor_index
        self.dcor_weight = dcor_weight 
        # here, y needs to be removed, and new parameters need to be added that can just be stored


# Step 2c. Looping through Dataset to get all Attributes needed for the GCN

In [ ]:
"""
Overall Data Present
1. Node Features
2. Edge Values
3. Distance Correlation
""" 
# To create the list of graphs 
dataset_list = []
for element, list_main in enumerate(list_of_openneuro_nodes):
    # assigning the list_main to the class, and executing the various methods
    patientval = graph_construction(list_main)
    patientval.create_edges()
    patientval.node_features_creation()
    patientval.distance_correlation()

    # This is the code segment where all of the values for the StaticGraphTemporalObject object are created. 

    # ____________________________________
    
    # Nodes 
    
    # ____________________________________
    """
    Current structure of the variable
    timestamp_wise_node_features = {
        "All node values for chunk 0": {
            "Fp1 for chunk 0": [...],
            "Fp2 for chunk 0": [...],
            ...
        },
        "All node values for chunk 1": {
            ...
        }
    }
    """ 
    # features is a list of np.arrays which have tensors, and each np array is one timeframe
    # For nodes, most of the heavy-lifting is done in the graph_construction class through the methods.
    # graph_construction.timestamp_wise_node_features -> is what the node features are stored in 

    features = []
    for timestamp_keys in graph_construction.timestamp_wise_node_features:
        timestamp_x = graph_construction.timestamp_wise_node_features[timestamp_keys].values()
        timestamp_x = list(timestamp_x)
        timestamp_x = np.array(timestamp_x, dtype=np.float32)
        timestamp_x = torch.from_numpy(timestamp_x)
        features.append(timestamp_x)
    print("---SIZE OF NODE FEATURES---")
    print(features[0].shape)
    #print("---NODE FEATURES XTH ELEMENT---")
    #print(features[0])
    #print(len(features[0]))
    print(len(features))

    # ____________________________________
    
    # Edges
    
    # ____________________________________
    
    # STEP 1. ------------------------------------------------------------------
    # defining all of the existing nodes
    # --------------------------------------------------------------------------
    
    nodes = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']

    edge_index_dict = {}
    # STEP 2. ------------------------------------------------------------------
    # making a loop to create the list of nodes and their value's which they are assigned, which applies throughout the algorithm
    # --------------------------------------------------------------------------
    
    counter = 0 
    for key_assignment in nodes:
        edge_index_dict.update({ key_assignment : counter })
        counter = counter + 1
    # ________________________________________________________________________
    # COMMENTED OUT(WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
    # print("------THE DICTIONARY OF NUMERICAL VALUES CONNECTED TO NODES------")
    # print(edge_index_dict)
    # ________________________________________________________________________

    # STEP 3. ------------------------------------------------------------------
    # getting the edge index (unordered)
    # --------------------------------------------------------------------------
    
    edge_index_unordered = []
    for edge_indice1, edge_indice2 in combinations(edge_index_dict.values(), 2):
        edge_indexxed_xandy = (edge_indice1, edge_indice2)
        edge_index_unordered.append(edge_indexxed_xandy)
    
    # normal edge index 
    normal_edge_index = edge_index_unordered
    # print("------UNORDERED INDEX OF EDGES------")
    # print(normal_edge_index)

    # STEP 4. ------------------------------------------------------------------
    # creating edge connecitivity strings
    # --------------------------------------------------------------------------
    
    edges = []
    for node_unit1, node_unit2 in combinations(nodes, 2 ):
        edge1 = (node_unit1, node_unit2)
        # edge2 = (node_unit2, node_unit1)
        edges.append(edge1)
        # edges.append(edge2)
    # print("-----EDGE-TO-EDGE CONNECTION VALUES-----")
    # print(edges)
    
    # STEP 5. ------------------------------------------------------------------
    # creating a combined dict for them to later be connected to connectivity_dict to produce edge_index and edge_values
    # --------------------------------------------------------------------------
    
    # print("-----COMBINED DICT OF BOTH-----")
    edge_connectivity_index = dict(zip(edges, normal_edge_index))
    # del normal_edge_index
    # print(edge_connectivity_index)
    # print(edges)

    # STEP 6. ------------------------------------------------------------------
    # GOAL - TO PROPERLY MAP EDGE INDEX AND EDGE WEIGHTS TO EACHOTHER. 
    # --------------------------------------------------------------------------
    
    ticker = 0 
    count = 0 
    edge_index_ordered = []
    edge_weights = []
    edge_weights_final = []
    # timestamps_of_edge_weights = {}
    for weight_chunks in range(0, patientval.chunks_length, 1):
        for keys in graph_construction.connectivity_dict:
            if ticker == 0: 
                edge_index_ordered.append(edge_connectivity_index[keys])
            edge_weights.append(graph_construction.connectivity_dict[keys][weight_chunks])
        edge_weights = np.array(edge_weights)
        edge_weights = torch.from_numpy(edge_weights)
        edge_weights_final.append(edge_weights)
            # timestamps_of_edge_weights.update({f"this is chunk {ticker} with the edge weights:" : edge_weights})
        ticker = ticker + 1 
        break
    
    # this is to transform the edge index into the ideal form
    
    edge_index = torch.tensor(edge_index_ordered)
    edge_index = torch.t(edge_index)
    
    # print("---SIZE OF EDGE INDEX---")
    # print(edge_index.shape)
    
    # this is to transform the edge weights into the ideal form
    # form needs to be [171, 1]
    # print("---SIZE OF EDGE WEIGHTS---")
    # print(edge_weights_final[0].shape)
    # print(edge_index.shape) -> the shape is proper as it is sopposed to be [2, ]
    # print("-----EDGE INDEX-------\n", edge_index) 
    # print("-----EDGE WEIGHTS-------\n", timestamps_of_edge_weights)

    # ____________________________________
    
    # Distance Correlation
    
    # ____________________________________
    
    """
    The purpose of this section is to transform the edge-to-edge relationships
    into numerical values which can later be used inside the custom GCN layer.
    Example:
    (('Fp1','Fp2'), ('Fp1','F3')) becomes:
    (0, 1, 0, 3), where:
    Fp1 = 0
    Fp2 = 1
    F3  = 3
    This allows the GCN to understand exactly which edges are participating
    in the distance correlation calculation.
    """
    
    # STEP 1. ------------------------------------------------------------------
    # getting the dcor index (unordered)
    # --------------------------------------------------------------------------
    
    dcor_index_unordered = []
    
    for edge1, edge2 in combinations(edges, 2):
        n1, n2 = edge1
        n3, n4 = edge2
        dcor_index_unordered.append(
            ( edge_index_dict[n1], edge_index_dict[n2], edge_index_dict[n3], edge_index_dict[n4] )
        )
    
    #print("------UNORDERED INDEX OF DCOR------")
    #print(dcor_index_unordered)

    # STEP 2. ------------------------------------------------------------------
    # creating dcor connectivity strings
    # --------------------------------------------------------------------------
    
    dcor_connections = []
    
    for dcor_indice1, dcor_indice2 in combinations(edges, 2):
        dcor_indexxed_xandy = (dcor_indice1, dcor_indice2)
    
        dcor_connections.append(dcor_indexxed_xandy)
    
    # print("-----DCOR CONNECTION VALUES-----")
    # print(dcor_connections)
    
    # STEP 3. ------------------------------------------------------------------
    # creating a combined dict for them to later be connected to edgetoedge_dict to produce dcor_index and dcor_weights
    # --------------------------------------------------------------------------
        
    dcor_connectivity_dict = dict(zip(dcor_connections, dcor_index_unordered ))
    # print("-----COMBINED DCOR DICT-----")
    
    # STEP 4. ------------------------------------------------------------------
    # GOAL - TO PROPERLY MAP DCOR INDEX AND DCOR WEIGHTS TO EACH OTHER.
    # --------------------------------------------------------------------------
    
    dcor_index_ordered = []
    dcor_weights = []
    
    # print(dcor_connectivity_dict[(('Fp1', 'Fp2'), ('F4', 'P3'))])
    for edge_keyvalues in graph_construction.edgetoedge_dict:
        dcor_index_ordered.append(dcor_connectivity_dict[edge_keyvalues])
        dcor_weights.append(graph_construction.edgetoedge_dict[edge_keyvalues])
    
    
    # turning into tensor format
    dcor_index = torch.tensor(dcor_index_ordered)
    dcor_index = torch.t(dcor_index)
    #print("---SIZE OF DCOR INDEX---")
    #print(dcor_index.shape)
    
    dcor_weights = torch.tensor(dcor_weights) # dtype=torch.float32
    #print("---SIZE OF DCOR WEIGHTS---")
    #print(dcor_weights.shape)
    
    # ____________________________________
    # COMMENTED OUT (WILL BE UNCOMMENTED WHEN NEEDED FOR TESTING PURPOSES)
    
    #print("-----DCOR INDEX-------")
    #print(dcor_index_ordered)
    
    #print("-----DCOR WEIGHTS-------")
    #print(dcor_weights)

    # ____________________________________
    
    # Y (Target values) for Training 
    
    # ____________________________________

    y_list = []
    for y_values in y_dict.values(): 
            y_list.append(y_values)
    y = [y_list[element]] * len(features)

    # This is the graph structure as a input which will be fed into the GCN 
    # Note: Tensors should be looked at vertically(where each column is what you are looking for)
    
    dataset = StaticGraphTemporalObject(
        edge_index,   
        edge_weights_final,
        features, 
        y,
        dcor_index, 
        dcor_weights 
    )
    dataset_list.append(dataset)        
    
    #print(dataset[0])
    print("One loop is done!")

---SIZE OF NODE FEATURES---
torch.Size([19, 7])
149
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
198
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
198
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
198
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
201
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
201
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
201
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
201
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
201
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
320
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
320
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
320
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
320
One loop is done!
---SIZE OF NODE FEATURES---
torch.Size([19, 7])
320
One loop is done!
---SIZE OF NODE FEAT

In [106]:
# confirmatory tests to see if the object is created correctly
print(dataset_list[4].features[0].shape)
print(dataset_list[4].edge_index.shape)
print(dataset_list[4].edge_weight[0].shape)
print(dataset_list[4].dcor_index.shape)
print(dataset_list[4].dcor_weight.shape)

torch.Size([19, 7])
torch.Size([2, 171])
torch.Size([171])
torch.Size([4, 14535])
torch.Size([14535])


AttributeError: 'StaticGraphTemporalObject' object has no attribute 'x'

# SEGMENT 2: THE T-GCN

# Step 1. Setting up the Dataset

In [21]:
random_seed = 43 # Will change later to being multiple seeds with are averaged
torch.manual_seed(1234567)
seed_everything(random_seed)
plt.style.use('dark_background')
accuracy_list = []

NameError: name 'seed_everything' is not defined

In [20]:
"""
All Components of the Dataset
1. Features (x)
2. Edge Index
3. Edge Weights
4. Dcor Index
5. Y values
6. Dcor Weights
"""
for datasets in dataset_list:
    train, test = temporal_signal_split(datasets, train_ratio=0.9)

# STEP 2. CREATING THE GCN SEGMENT OF THE MODEL

In [18]:
# Step 2. Creating the Model
"""
Note, this segment of the model is adaptable, as the base of the model is PyTorch Geometric. However, this model
has the most advanced infrastructure for the function, it is possible to edit this model for future approaches in 
the concept aswell. Though in due time this model will become very specialized, but this version will remain as archived and will also be opensource
Pytorch GCN Fundamental Conv Layer: https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GCNConv.html?utm_source=chatgpt.com#torch_geometric.nn.conv.GCNConv.backward 
Pytorch GNN's Implementation: https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#torch_geometric.nn.conv.GCNConv
"""

#Creating GCNConvolutional as a Child class of GCNConv which will have custom properties that will allow the Algorithm to function
class GCNConvolutional(GCNConv):
    
    def edgenode_effect():
        # dcor_weight.view(-1,1) * x_j
         
    def propagate(edge_index, x, edge_weight, dcor_index, dcor_weight):
        
    def message(x_j: Tensor, x_i: Tensor): 
        return edgenode_effect(x_i), x_j if edge_weight is None else (x_j := edge_weight.view(-1, 1) * x_j)
        # edge-to-edge correlation 
  
    def aggregate():
    
    def update(): 
        # arbitrary tensors: since its a seperate variable not under the presets of other variables
        # PyG has, we can just pass it in, and put alot of the logic in the GCN's layers itself

    def forward(): 


IndentationError: expected an indented block after function definition on line 13 (2397329276.py, line 17)

# STEP 4. CREATING THE FULL T-GCN

In [20]:
"""
hello there
"""

class TGCN2(torch.nn.Module):
    r"""An implementation THAT SUPPORTS BATCHES of the Temporal Graph Convolutional Gated Recurrent Cell.
    For details see this paper: `"T-GCN: A Temporal Graph ConvolutionalNetwork for
    Traffic Prediction." <https://arxiv.org/abs/1811.05320>`_

    Args:
        in_channels (int): Number of input features.
        out_channels (int): Number of output features.
        batch_size (int): Size of the batch.
        improved (bool): Stronger self loops. Default is False.
        cached (bool): Caching the message weights. Default is False.
        add_self_loops (bool): Adding self-loops for smoothing. Default is True.
    """

    def __init__(self, in_channels: int, out_channels: int, 
                 batch_size: int,  # this entry is unnecessary, kept only for backward compatibility
                 improved: bool = False, cached: bool = False, 
                 add_self_loops: bool = True):
        super(TGCN2, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.improved = improved
        self.cached = cached
        self.add_self_loops = add_self_loops
        self.batch_size = batch_size  # not needed
        self._create_parameters_and_layers()

    def _create_update_gate_parameters_and_layers(self):
        self.conv_z = GCNConv(in_channels=self.in_channels,  out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_z = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_reset_gate_parameters_and_layers(self):
        self.conv_r = GCNConv(in_channels=self.in_channels, out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_r = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_candidate_state_parameters_and_layers(self):
        self.conv_h = GCNConv(in_channels=self.in_channels, out_channels=self.out_channels, improved=self.improved,
                              cached=self.cached, add_self_loops=self.add_self_loops )
        self.linear_h = torch.nn.Linear(2 * self.out_channels, self.out_channels)

    def _create_parameters_and_layers(self):
        self._create_update_gate_parameters_and_layers()
        self._create_reset_gate_parameters_and_layers()
        self._create_candidate_state_parameters_and_layers()

    def _set_hidden_state(self, X, H):
        if H is None:
            # can infer batch_size from X.shape, because X is [B, N, F]
            H = torch.zeros(X.shape[0], X.shape[1], self.out_channels).to(X.device) #(b, 207, 32)
        return H

    def _calculate_update_gate(self, X, edge_index, edge_weight, H):
        Z = torch.cat([self.conv_z(X, edge_index, edge_weight), H], axis=2) # (b, 207, 64)
        Z = self.linear_z(Z) # (b, 207, 32)
        Z = torch.sigmoid(Z)

        return Z

    def _calculate_reset_gate(self, X, edge_index, edge_weight, H):
        R = torch.cat([self.conv_r(X, edge_index, edge_weight), H], axis=2) # (b, 207, 64)
        R = self.linear_r(R) # (b, 207, 32)
        R = torch.sigmoid(R)

        return R

    def _calculate_candidate_state(self, X, edge_index, edge_weight, H, R):
        H_tilde = torch.cat([self.conv_h(X, edge_index, edge_weight), H * R], axis=2) # (b, 207, 64)
        H_tilde = self.linear_h(H_tilde) # (b, 207, 32)
        H_tilde = torch.tanh(H_tilde)

        return H_tilde

    def _calculate_hidden_state(self, Z, H, H_tilde):
        H = Z * H + (1 - Z) * H_tilde   # # (b, 207, 32)
        return H

    def forward(self,X: torch.FloatTensor, edge_index: torch.LongTensor, edge_weight: torch.FloatTensor = None,
                H: torch.FloatTensor = None ) -> torch.FloatTensor:
        """
        Making a forward pass. If edge weights are not present the forward pass
        defaults to an unweighted graph. If the hidden state matrix is not present
        when the forward pass is called it is initialized with zeros.

        Arg types:
            * **X** *(PyTorch Float Tensor)* - Node features.
            * **edge_index** *(PyTorch Long Tensor)* - Graph edge indices.
            * **edge_weight** *(PyTorch Long Tensor, optional)* - Edge weight vector.
            * **H** *(PyTorch Float Tensor, optional)* - Hidden state matrix for all nodes.

        Return types:
            * **H** *(PyTorch Float Tensor)* - Hidden state matrix for all nodes.
        """
        H = self._set_hidden_state(X, H)
        Z = self._calculate_update_gate(X, edge_index, edge_weight, H)
        R = self._calculate_reset_gate(X, edge_index, edge_weight, H)
        H_tilde = self._calculate_candidate_state(X, edge_index, edge_weight, H, R)
        H = self._calculate_hidden_state(Z, H, H_tilde) # (b, 207, 32)
        return H

In [22]:
"""
The GCN will combine the information from both 

# Training
# Validation
# Testing 
"""

model = TGCN2(16, 32, 1)

x = self.gcn(...)
x = F.relu
x = self.gcn(...)
x = F.relu
x = self.gru(...)
x = global_mean_pool(x, batch)
x = self.classifier(x)

model.train()

model.eval()
# From this stage, we can get closer to computing accuracy

NameError: name 'self' is not defined

In [ ]:
# ARCHIVED SEGMENT
    
# Convoltuional Layer is GCNConv
model = Sequential(
    'x, edge_index', [
    # layer 1 or first propogation
    (GCNConvolutional(in_channels, 64, ADD_AGGREGGATION_LAYER), 'x, edge_index -> x'),

    # ReLu activation function 
    ReLU(inplace=True),

    # layer 2 on second propogation 
    (GCNConvolutional(64, 64, ADD_AGGREGGATION_LAYER), 'x, edge_index -> x'),

    # ReLu activation function 
    ReLU(inplace=True),

    # Applying Lineararity to the Results 
    Linear(64, out_channels),
]
                  )

# SEGMENT 3: INFERENCE(ALZHEIMER'S PROBABILITY FOR JUST ONE PERSON) 
80% Weightage

# SEGMENT 4 - AI-BASED ALZHEIMER'S ANALYSIS BASED ON QUESTIONAIRE (MMSE Based) 
20% WEIGHTAGE

In [ ]:
# Imports

In [ ]:
# Code

# SEGMENT 5 - FINAL COMPILER (DETERMINES WHETHER THE INDIVIDUAL HAS ALZHEIMER'S

In [ ]:
"""
This is the segment of the algorithm which will determine whether the individual has alzheimer's or a neurodegenerative condition in general.
In the first iteration it will give a determination, and will be coded to also provide a confidence rating. This segment will take as input
1. The EEG Segment -> The output from this segment is taken into account as 80% of the weightage"
2. The Questionaire -> The output from this segment is taken into account as 20% of the weightage
"""

In [51]:
# this code is for keeping the cell continously active to bypass the 40-minute idle timeout
import time
try:
    while True:
        time.sleep(60)  # Pauses for 1 minute
except KeyboardInterrupt:
    print("Loop stopped manually.")

Loop stopped manually.
